# 模型分数分箱 — 生产代码

按照操作指南中的字段口径，实现完整的等频分箱、指标计算、合并相邻箱、阈值曲线和策略阈值确定。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.stats import chi2_contingency
import statsmodels.api as sm

# 显示设置
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# ============================================================
# 1. 加载数据
# ============================================================

# 主表：样本表
sample = pd.read_csv('res/sample.csv')
print(f"sample: {sample.shape}")

# 申请信息表（已按 sample 对齐）
app = pd.read_csv('res/application_info.csv')
# BOM 头处理
app.columns = [c.lstrip('ï»¿') for c in app.columns]
print(f"app: {app.shape}")

# 模型分表
mlt_score = pd.read_csv('res/aus_old_risk_bid_mltmodel_v1_2_20260325_lgb_score.csv')
apply_score = pd.read_csv('res/aus_old_risk_apply_appmodel_v20260318_v1_2_lgb_score.csv')
txn_score = pd.read_csv('res/aus_old_risk_bid_submodel_v20260323_v1_2_txn_lgb_score.csv')
print(f"mlt_score: {mlt_score.shape}")
print(f"apply_score: {apply_score.shape}")
print(f"txn_score: {txn_score.shape}")

In [ ]:
# ============================================================
# 2. 拼接主表（与 scr/sample_wide.sql 口径一致）
# ============================================================

# 以 sample 为底表，左连接各表
df = sample.merge(app, on=['application_id', 'user_id'], how='left')

# 模型分：只保留 score 列，重命名
mlt_col = 'aus_old_risk_bid_mltmodel_v1_2_v20260325_lgb_score'
apply_col = 'aus_old_risk_apply_appmodel_v20260318_v1_2_lgb_score'

df = df.merge(
    mlt_score[['application_id', mlt_col]],
    on='application_id', how='left'
).rename(columns={mlt_col: 'score_mlt'})

df = df.merge(
    apply_score[['application_id', apply_col]],
    on='application_id', how='left'
).rename(columns={apply_col: 'score_apply'})

# 交易特征子模型：保留 score + 21 个交易特征
txn_feature_cols = [c for c in txn_score.columns
                    if c not in ('application_id', 'user_id', 'sample_datetime',
                                 'feature_error', 'send_time',
                                 'transaction_date_max', 'balance_date_max')]
df = df.merge(
    txn_score[['application_id'] + txn_feature_cols],
    on='application_id', how='left'
)

# 只保留分箱需要的字段（与 sample_wide.sql 一致）
keep_cols = [
    'application_id', 'user_id',
    'application_time', 'application_date', 'application_month',
    'score_mlt', 'score_apply',
    # 交易子模型 score + 特征
    *txn_feature_cols,
    # 笔数逾期
    'duedate_1m_30', 'duedate_3m_30',
    # 金额逾期
    'principal', 'estimate_principal_remaining_mob1',
    'estimate_principal_remaining_mob3',
    'dpd_days_ever_mob1', 'dpd_days_ever_mob3',
    # 漏斗
    'application_status', 'status',
    # 分析维度
    'LTI', 'PTI', 'NSTI',
    'application_tag', 'user_tag', 'loan_tag',
]

# 只保留实际存在的列
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols]

print(f"df: {df.shape}")
print(f"columns ({len(keep_cols)}): {df.columns.tolist()}")
print(f"\n缺失率概览:\n{df.isnull().mean().round(4).sort_values(ascending=False).head(15)}")